# 阶段 5：原作者的 sft 1.5B 转成 GGUF（Colab，CPU 就够）

把 `MyDrive/COMLLM/reference_model/qwen2.5-1.5b-mec-grpo-final-all/` 转成 **Q8_0** 的 GGUF，放到 Jetson 上量"论文原版做法"的真实延迟（实验 2 的 Y），
也让论文原版 1.5B 能进闭环（可选）。用的 llama.cpp 和 Jetson 上是同一个提交（5aeac37a3），和阶段 3 训练 notebook 导出 GGUF 的做法一样。

不用改任何东西，直接「代码执行程序 → 全部运行」。**不需要 GPU**：运行时选 CPU 就行，省额度。

| 步骤 | 做什么 | 预计 |
|---|---|---|
| 第 1 格 | 装转换脚本要的库，挂载 Drive，找模型文件夹 | 约 2 分钟（估算） |
| 第 2 格 | 下载 llama.cpp 源码，切到 5aeac37a3 | 约 1 分钟（估算） |
| 第 3 格 | 转 Q8_0，算校验值，写到 Drive | 约 3–6 分钟（估算） |

**结果**：`MyDrive/COMLLM/s5/gguf/comllm-paper-grpo-1.5b-q8_0.gguf`（约 1.6–1.9 GB，估算）和同目录的 `convert_grpo.json`（校验值、源模型、llama.cpp 提交）。

**止损线**：任何一格报错就停，截图发到对话里；总共超过 30 分钟还没转完也停。转不出来的话，实验 2 的论文原版一行改用阶段 2 的做法（未训练 1.5B + 论文格式提示词，延迟一样只差答案内容），在 README 里写明。


In [ ]:
# 第 1 格：装库、挂载 Drive、找模型
import subprocess, sys, os, json, time, hashlib
from datetime import datetime, timezone
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.20.2", "sentencepiece"], check=True)

from google.colab import drive
drive.mount("/content/drive")
SRC = "/content/drive/MyDrive/COMLLM/reference_model/qwen2.5-1.5b-mec-grpo-final-all"
OUT = "/content/drive/MyDrive/COMLLM/s5/gguf"
LLAMA_CPP_COMMIT = "5aeac37a3"          # 和 Jetson 上的 llama.cpp 同一个提交
assert os.path.exists(os.path.join(SRC, "config.json")), "找不到模型文件夹：" + SRC
os.makedirs(OUT, exist_ok=True)
print("源模型：", SRC)
print("文件：", sorted(os.listdir(SRC)))


In [ ]:
# 第 2 格：下载 llama.cpp 源码，切到和 Jetson 同一个提交（转换脚本只用 Python，不用编译）
def run_quiet(cmd, what):
    """跑一个外部命令；成功只打一行，失败打最后 40 行再报错停下。"""
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join((r.stdout + r.stderr).splitlines()[-40:]))
        raise RuntimeError(f"{what} 失败（返回码 {r.returncode}），上面是最后 40 行输出")
    print(f"{what} 完成，用时 {time.time() - t0:.0f} 秒")


LCPP = "/content/llama.cpp"
if not os.path.isdir(LCPP):
    run_quiet(["git", "clone", "-q", "https://github.com/ggml-org/llama.cpp", LCPP], "下载 llama.cpp 源码")
run_quiet(["git", "-C", LCPP, "checkout", "-q", LLAMA_CPP_COMMIT], "切到提交 " + LLAMA_CPP_COMMIT)


In [ ]:
# 第 3 格：转 Q8_0 → 算 SHA-256 → 写 convert_grpo.json
# 先写到 Colab 本地盘再复制到 Drive：直接写 Drive 有时很慢
local = "/content/comllm-paper-grpo-1.5b-q8_0.gguf"
run_quiet([sys.executable, f"{LCPP}/convert_hf_to_gguf.py", SRC, "--outtype", "q8_0", "--outfile", local], "转 Q8_0")
h = hashlib.sha256()
with open(local, "rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
sha = h.hexdigest()
dst = os.path.join(OUT, os.path.basename(local))
run_quiet(["cp", local, dst], "复制到 Drive")
info = {"source": SRC, "out": dst, "outtype": "q8_0", "llama_cpp": LLAMA_CPP_COMMIT, "sha256": sha,
        "size_mb": round(os.path.getsize(local) / 1e6), "utc": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}
with open(os.path.join(OUT, "convert_grpo.json"), "w") as f:
    json.dump(info, f, ensure_ascii=False, indent=2)
print(json.dumps(info, ensure_ascii=False, indent=2))
print("下一步：把这个 .gguf 下载到 Mac，再 scp 到 Jetson 的 ~/models/（运行步骤文档里有命令）")
